# KG1 V80 MEGA V3 - TODOS OS FIXES CONSOLIDADOS

## Execução ultra-simplificada
**APENAS 1 célula. Execute e espera ~3h.**

O que faz automaticamente:
1. Uninstall torchcodec/torchao/torchdata (Colab pre-installed, conflita com torch 2.5)
2. Install torch 2.5.1+cu124 via wheels diretos (bypass --index-url issues)
3. Install mamba-ssm 2.2.4 + causal-conv1d 1.5.0.post8 (torch 2.5 ABI)
4. Install transformers/peft/trl/accelerate/datasets/bitsandbytes
5. Install Unsloth com --no-deps (não upgrade torch)
6. Verify via child process (torch 2.5 clean import)
7. Download colab_mega_v80.py do GitHub
8. Executa training em subprocess:
   - Dataset dgxchen v7 EXACT (problem_ids_matched.csv, 7830 rows)
   - Model Nemotron-3-Nano-30B-A3B-BF16 (cached)
   - LoRA r=32 alpha=32 dropout=0, 8 targets SEM lm_head
   - max_length=**3072** (p99 safe, otimizado H100 80GB)
   - Train 1 epoch 245 steps
   - Save adapter + Build submission.zip + HF upload + Kaggle submit

**Tempo total**: ~3-4h (com cache) ou ~4-5h (cold start)

## Todos os 13 fixes aplicados

### 7 divergências dgxchen v7 revertidas:
1. Dataset `problem_ids_matched.csv` (não less_cot.csv)
2. attn_implementation='eager' (não sdpa)
3. LoRA 8 targets SEM lm_head
4. num_train_epochs=1 (não 2)
5. max_grad_norm=1e9 (efetivamente disabled)
6. gradient_checkpointing=True + use_reentrant=False
7. formatting_func no trainer com conversation wrap

### 4 fixes execução (descobertos hoje 22/04):
8. mamba-ssm + causal-conv1d install explicit (NemotronH requer)
9. torch 2.5.1 pin via WHEELS DIRETOS (bypass --index-url bug Colab cp312)
10. dataloader_num_workers=0 (prev pickle CudaDeviceProperties error)
11. Unsloth --no-deps + uninstall torchcodec (prev ABI mismatch torch 2.11)

### 2 fixes performance (descobertos no primeiro run):
12. MAX_SEQ_LEN=**3072** (não 4096) — evita gradient offloading, 4x speedup
13. PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True (anti-fragmentation)

## Credenciais (Colab Secrets 🔒)
Adicione os 3 na barra lateral (ícone cadeado):
- `HF_KEY` = (seu HF token DEV atual)
- `KAGGLE_USERNAME` = felipe1983
- `KAGGLE_KEY` = (do kaggle.json)

## Hardware
- **Recomendado**: Colab Pro+ **H100 80GB HBM3**
- Também funciona: A100 80GB
- **Não funciona**: T4/L4/A10 (VRAM insuficiente para Nemotron-30B + MoE LoRA)

## Expected outcome
Score Kaggle: **0.84-0.85** (replica dgxchen v7 EXACT com 0.85 LB verificado 22/04/2026)

## Credenciais (Colab Secrets - icone cadeado no painel esquerdo)
Adicione estes 3 secrets no Colab (Runtime > Secrets):
- `HF_KEY` = (seu HF token DEV)
- `KAGGLE_USERNAME` = felipe1983
- `KAGGLE_KEY` = (do kaggle.json)

## Hardware
- **Recomendado**: Colab Pro+ H100 80GB HBM3
- Também funciona: A100 80GB
- **Não funciona**: T4, L4, A10 (VRAM insuficiente para 30B BF16)


In [1]:
# V80 MEGA CELL - install + download + train + submit (tudo em 1 cell)
# ATENCAO: espera ~3-4h para terminar. Output streams aqui live.
import subprocess, sys, os, json, urllib.request
from pathlib import Path

print('=' * 70)
print('V80 MEGA CELL - dgxchen v7 EXACT, 1 cell, no restart')
print('=' * 70)

# ============ 1. Load Colab secrets ============
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_KEY')
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
except Exception:
    hf_key = os.environ.get('HF_KEY') or os.environ.get('HF_TOKEN')
    kaggle_user = os.environ.get('KAGGLE_USERNAME')
    kaggle_key = os.environ.get('KAGGLE_KEY')

assert hf_key, 'HF_KEY missing - add in Colab Secrets (cadeado esquerda)'
assert kaggle_user and kaggle_key, 'KAGGLE_USERNAME / KAGGLE_KEY missing'

os.environ['HF_TOKEN'] = hf_key
os.environ['HF_KEY'] = hf_key
os.environ['KAGGLE_USERNAME'] = kaggle_user
os.environ['KAGGLE_KEY'] = kaggle_key

# kaggle.json
kpath = Path.home() / '.kaggle' / 'kaggle.json'
kpath.parent.mkdir(parents=True, exist_ok=True)
kpath.write_text(json.dumps({'username': kaggle_user, 'key': kaggle_key}))
kpath.chmod(0o600)

print(f'HF token: ...{hf_key[-8:]}')
print(f'Kaggle user: {kaggle_user}')

# ============ 2. GPU check ============
import torch
assert torch.cuda.is_available(), 'CUDA/GPU required (use Colab H100 or A100)'
d = torch.cuda.get_device_properties(0)
total_gb = d.total_memory / 1024**3
print(f'GPU: {d.name} {total_gb:.1f}GB')
assert total_gb >= 38, f'Need 40GB+ GPU, got {total_gb:.1f}GB'

# ============ 3. Install torch 2.5.1 + mamba-ssm + ML stack ============
print()
print('Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...')
print('Expected: ~10 min (torch wheel download + install)')


def sh(cmd, timeout=900, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        print(f'  FAIL: {" ".join(cmd[:6])}')
        print(f'  stderr: {r.stderr[-500:]}')
        raise RuntimeError('Command failed')
    return r


# Uninstall torch + Colab pre-installed torch.* packages (torch 2.11 ABI conflicts)
print('  Uninstalling existing torch + Colab pre-installed torchcodec/torchao/etc...')
for pkg in ['torch', 'torchvision', 'torchaudio',
            'torchcodec', 'torchao', 'torchdata', 'torchtune', 'torchsummary']:
    for i in range(5):
        r = sh([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], check=False)
        if 'Successfully uninstalled' not in r.stdout:
            break

# Install torch 2.5.1+cu124 via DIRECT wheel (bypass index-url resolution issues)
print('  Installing torch 2.5.1+cu124 (direct wheels)...')
for url in [
    'https://download.pytorch.org/whl/cu124/torch-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchvision-0.20.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchaudio-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
]:
    sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', url])

# torch runtime deps
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'filelock', 'jinja2', 'networkx', 'fsspec', 'sympy>=1.13', 'typing-extensions'])

# Mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)
print('  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...')
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])

# ML stack
print('  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...')
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<4.58', 'peft>=0.14,<0.18', 'trl>=0.14,<0.26',
    'accelerate>=1.0,<2.0', 'datasets>=3.2,<5',
    'bitsandbytes', 'huggingface_hub', 'safetensors', 'einops',
    'sentencepiece', 'pandas', 'kagglehub', 'einx'])

# Unsloth --no-deps (no torch upgrade)
print('  Installing unsloth + unsloth_zoo (--no-deps)...')
sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'unsloth', 'unsloth_zoo', 'xformers', 'tyro', 'hf_transfer'], check=False)

# Verify via child process (child = fresh torch 2.5 import from disk)
print()
print('Verifying install via child process (torch 2.5 clean import)...')
r = subprocess.run([sys.executable, '-c', (
    "import torch, mamba_ssm; "
    "print(f'child: torch={torch.__version__} cuda={torch.version.cuda}'); "
    "print(f'child: mamba_ssm={mamba_ssm.__version__}'); "
    "from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn; "
    "from unsloth import FastLanguageModel; "
    "print('child: ALL IMPORTS OK')"
)], capture_output=True, text=True, timeout=120)
print(r.stdout)
if r.returncode != 0:
    print('child stderr:', r.stderr[-500:])
    raise RuntimeError('Child process verify failed - deps broken')

# ============ 4. Download training script from GitHub ============
print()
print('Downloading training script from GitHub (colab_mega_v80.py)...')
SCRIPT_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts/colab_mega_v80.py'
SCRIPT_PATH = '/content/colab_mega_v80.py'

try:
    urllib.request.urlretrieve(SCRIPT_URL, SCRIPT_PATH)
    sz = os.path.getsize(SCRIPT_PATH)
    print(f'  Downloaded: {SCRIPT_PATH} ({sz/1024:.1f} KB)')
except Exception as e:
    print(f'  GitHub download failed: {e}')
    print('  Trying HF dataset repo fallback...')
    from huggingface_hub import hf_hub_download
    local = hf_hub_download(
        repo_id='felipesp1983/kg1-nemotron-training',
        filename='scripts/colab_mega_v80.py',
        repo_type='dataset',
        token=hf_key,
    )
    import shutil
    shutil.copy2(local, SCRIPT_PATH)
    print(f'  HF fallback OK: {SCRIPT_PATH} ({os.path.getsize(SCRIPT_PATH)/1024:.1f} KB)')

# ============ 5. Execute training in subprocess (child has clean torch 2.5) ============
print()
print('=' * 70)
print('Starting V80 training in child process (streams output live)')
print('Expected: ~3-4h (download if cold + train 245 steps + submit)')
print('=' * 70)
print()

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # anti-fragmentation
env['MAX_SEQ_LEN'] = '3072'  # p99 safe, H100 80GB fit, ~40s/step (vs 2.75min com 4096)

proc = subprocess.Popen(
    [sys.executable, '-u', SCRIPT_PATH],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.wait()

print()
print('=' * 70)
print(f'V80 MEGA DONE - return code {proc.returncode}')
if proc.returncode == 0:
    print('SUCCESS: training finished, submission made')
    print('Check Kaggle score: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions')
else:
    print('FAILED: see output above for diagnosis')
print('=' * 70)


V80 MEGA CELL - dgxchen v7 EXACT, 1 cell, no restart
HF token: ...ifYYkxHG
Kaggle user: felipe1983
GPU: NVIDIA H100 80GB HBM3 79.2GB

Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...
Expected: ~10 min (torch wheel download + install)
  Uninstalling existing torch + Colab pre-installed torchcodec/torchao/etc...
  Installing torch 2.5.1+cu124 (direct wheels)...
  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...
  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...
  Installing unsloth + unsloth_zoo (--no-deps)...

Verifying install via child process (torch 2.5 clean import)...
child: torch=2.5.1+cu124 cuda=12.4
child: mamba_ssm=2.2.4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
child: ALL IMPORTS OK


  Downloaded: /content/colab_mega_v80.py (18.8 KB)

Starting V80 training in child process (streams output live)
Expected: ~3-4h (download if cold + tra

KeyboardInterrupt: 

In [2]:
# V3.1 PATCH - restart com paged_adamw_8bit (3-4x speedup real)
import subprocess, sys, os, urllib.request, time, torch, gc

print('=' * 70)
print('V3.1 PATCH - optim=paged_adamw_8bit (economiza 5GB VRAM)')
print('AdamW FP32 7GB -> AdamW 8bit 1.8GB -> Unsloth desliga offload -> 3-4x speedup')
print('Expected ETA: 19h -> ~2.5h')
print('=' * 70)

# 1. Kill V3 subprocess
subprocess.run(['pkill', '-9', '-f', 'colab_mega_v80.py'], capture_output=True)
time.sleep(3)
print('V3 subprocess killed')

# 2. Clear GPU
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f}GB')

# 3. Download V3.1 script (commit 1c73ed5)
print('\nDownloading V3.1 script (paged_adamw_8bit fix)...')
url = f'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts/colab_mega_v80.py?t={int(time.time())}'
urllib.request.urlretrieve(url, '/content/colab_mega_v80.py')

# Verify fix
with open('/content/colab_mega_v80.py') as f:
    content = f.read()
assert 'paged_adamw_8bit' in content, 'V3.1 fix missing'
print('V3.1 fix verified (paged_adamw_8bit present)')

# 4. Run subprocess
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['MAX_SEQ_LEN'] = '3072'

print('\n' + '=' * 70)
print('V3.1 training starting (ETA ~2.5h)')
print('Cache: install + model 50GB + dataset (all preserved)')
print('=' * 70)
print()

proc = subprocess.Popen(
    [sys.executable, '-u', '/content/colab_mega_v80.py'],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.wait()

print(f'\nV3.1 DONE - rc={proc.returncode}')

V3.1 PATCH - optim=paged_adamw_8bit (economiza 5GB VRAM)
AdamW FP32 7GB -> AdamW 8bit 1.8GB -> Unsloth desliga offload -> 3-4x speedup
Expected ETA: 19h -> ~2.5h
V3 subprocess killed
GPU free: 78.7GB

V3.1 fix verified (paged_adamw_8bit present)

V3.1 training starting (ETA ~2.5h)
Cache: install + model 50GB + dataset (all preserved)


[CHILD] torch=2.5.1+cu124  cuda=12.4
[CHILD] GPU: NVIDIA H100 80GB HBM3 79.2GB
[CHILD] mamba_ssm=2.2.4

STEP 2/7: Download dgxchen dataset (problem_ids_matched.csv)
Dataset cached: 46.1MB
Dataset path: /content/kg1_data/problem_ids_matched.csv
Rows: 7830
Columns: ['id', 'prompt', 'answer', 'type', 'generated_cot']
Type distribution:
  bit_manipulation: 1754
  cipher: 1656
  unit_conversion: 1070
  gravity: 1055
  numeral: 730
  equation_numeric_deduce: 658
  cryptarithm_deduce: 627
  cryptarithm_guess: 154
  equation_numeric_guess: 126

STEP 3/7: Download Nemotron-3-Nano-30B-A3B-BF16 base model
Model cached at: /root/.cache/kagglehub/models/metric/nemotr

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# V80 MEGA CELL FINAL - 14 fixes inline - dgxchen v7 EXACT replica (0.85 LB target)
# 1 célula, zero restart, zero dependência de GitHub
# Expected: ~3h H100 80GB -> score 0.84-0.85
# =============================================================================
import subprocess, sys, os, json, gc, time, urllib.request
from pathlib import Path

print('=' * 70)
print('V80 MEGA FINAL - 14 fixes consolidados')
print('=' * 70)

# ===== PART 1: SECRETS =====
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_KEY')
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
except Exception:
    hf_key = os.environ.get('HF_KEY') or os.environ.get('HF_TOKEN')
    kaggle_user = os.environ.get('KAGGLE_USERNAME')
    kaggle_key = os.environ.get('KAGGLE_KEY')

assert hf_key, 'HF_KEY missing - add in Colab Secrets (cadeado)'
assert kaggle_user and kaggle_key, 'KAGGLE_USERNAME / KAGGLE_KEY missing'

os.environ['HF_TOKEN'] = hf_key
os.environ['HF_KEY'] = hf_key
os.environ['KAGGLE_USERNAME'] = kaggle_user
os.environ['KAGGLE_KEY'] = kaggle_key

kpath = Path.home() / '.kaggle' / 'kaggle.json'
kpath.parent.mkdir(parents=True, exist_ok=True)
kpath.write_text(json.dumps({'username': kaggle_user, 'key': kaggle_key}))
kpath.chmod(0o600)

print(f'HF token: ...{hf_key[-8:]}')
print(f'Kaggle user: {kaggle_user}')

# ===== PART 2: GPU CHECK =====
import torch
assert torch.cuda.is_available(), 'GPU required'
d = torch.cuda.get_device_properties(0)
total_gb = d.total_memory / 1024**3
print(f'GPU: {d.name} {total_gb:.1f}GB')
assert total_gb >= 38, f'Need 40GB+ GPU'

# ===== PART 3: INSTALL DEPS =====
print()
print('Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...')

def sh(cmd, timeout=900, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        print(f'  FAIL: {" ".join(cmd[:6])}')
        print(r.stderr[-500:])
        raise RuntimeError('Command failed')
    return r

# FIX 11: Uninstall Colab pre-installed torch + torchcodec/torchao/etc (torch 2.11 ABI conflict)
print('  Uninstalling torch + Colab pre-installed torchcodec/torchao/torchdata...')
for pkg in ['torch', 'torchvision', 'torchaudio',
            'torchcodec', 'torchao', 'torchdata', 'torchtune', 'torchsummary']:
    for _ in range(5):
        r = sh([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], check=False)
        if 'Successfully uninstalled' not in r.stdout:
            break

# FIX 9: Install torch 2.5.1+cu124 via DIRECT WHEEL URLs (bypass --index-url resolution issues)
print('  Installing torch 2.5.1+cu124 (direct wheels)...')
for url in [
    'https://download.pytorch.org/whl/cu124/torch-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchvision-0.20.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
    'https://download.pytorch.org/whl/cu124/torchaudio-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl',
]:
    sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', url])

sh([sys.executable, '-m', 'pip', 'install', '-q',
    'filelock', 'jinja2', 'networkx', 'fsspec', 'sympy>=1.13', 'typing-extensions'])

# FIX 8: Install mamba-ssm + causal-conv1d wheels (torch 2.5 ABI, cp312+cu12)
print('  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...')
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/state-spaces/mamba/releases/download/v2.2.4/'
    'mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])
sh([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/'
    'causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'])

# ML stack (without upgrading torch)
print('  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...')
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<4.58', 'peft>=0.14,<0.18', 'trl>=0.14,<0.26',
    'accelerate>=1.0,<2.0', 'datasets>=3.2,<5',
    'bitsandbytes', 'huggingface_hub', 'safetensors', 'einops',
    'sentencepiece', 'pandas', 'kagglehub', 'einx'])

# FIX 11: Unsloth --no-deps (previne upgrade accidental torch para 2.11)
print('  Installing unsloth + unsloth_zoo (--no-deps)...')
sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'unsloth', 'unsloth_zoo', 'xformers', 'tyro', 'hf_transfer'], check=False)

# Verify child process sees torch 2.5 clean
print()
print('Verifying child process (torch 2.5 clean import)...')
r = subprocess.run([sys.executable, '-c', (
    "import torch, mamba_ssm; "
    "print(f'child: torch={torch.__version__} cuda={torch.version.cuda}'); "
    "print(f'child: mamba_ssm={mamba_ssm.__version__}'); "
    "from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn; "
    "from unsloth import FastLanguageModel; "
    "print('child: ALL IMPORTS OK')"
)], capture_output=True, text=True, timeout=180)
print(r.stdout)
if r.returncode != 0:
    print('child stderr:', r.stderr[-500:])
    raise RuntimeError('Deps broken')

# ===== PART 4: WRITE TRAINING SCRIPT INLINE =====
TRAIN_SCRIPT = r'''#!/usr/bin/env python3
"""V80 dgxchen v7 EXACT - training script with all 14 fixes."""
import sys, os, gc, re, math, time, json, random, shutil, zipfile, datetime, subprocess
from pathlib import Path
from collections import defaultdict, deque

for s in (sys.stdout, sys.stderr):
    if hasattr(s, "reconfigure"): s.reconfigure(encoding="utf-8", errors="replace")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TQDM_DISABLE", "1")

import torch
print(f"\n[CHILD] torch={torch.__version__}  cuda={torch.version.cuda}")
assert torch.__version__.startswith("2.5"), f"Need torch 2.5, got {torch.__version__}"
d = torch.cuda.get_device_properties(0)
print(f"[CHILD] GPU: {d.name} {d.total_memory/1024**3:.1f}GB")

import mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
print(f"[CHILD] mamba_ssm={mamba_ssm.__version__}")

SEED = 42
random.seed(SEED); import numpy as np; np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_KEY")
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", 3072))  # FIX 12: p99 safe
print(f"[CHILD] MAX_SEQ_LEN={MAX_SEQ_LEN}")

# ===== STEP 2: Dataset =====
print("\n" + "=" * 70)
print("STEP 2/7: Download dgxchen dataset (FIX 1: problem_ids_matched.csv)")
print("=" * 70)

DATA_DIR = Path("/content/kg1_data")
DATA_DIR.mkdir(exist_ok=True)
target_csv = DATA_DIR / "problem_ids_matched.csv"

if target_csv.exists() and target_csv.stat().st_size > 40_000_000:
    print(f"Dataset cached: {target_csv.stat().st_size/(1024**2):.1f}MB")
else:
    r = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "dgxchen/nemotron-cot-tong",
         "-p", str(DATA_DIR), "--unzip"],
        capture_output=True, text=True, timeout=300)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"Kaggle download failed: {r.stderr[-500:]}")

import pandas as pd
df = pd.read_csv(target_csv)
print(f"Rows: {len(df)}  Columns: {list(df.columns)}")
for t, n in df["type"].value_counts().items():
    print(f"  {t}: {n}")

# ===== STEP 3: Base model =====
print("\n" + "=" * 70)
print("STEP 3/7: Nemotron-3-Nano-30B-A3B-BF16 base model")
print("=" * 70)

import kagglehub
MODEL_CACHE = "/root/.cache/kagglehub/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
if Path(MODEL_CACHE).exists() and len(list(Path(MODEL_CACHE).iterdir())) > 5:
    MODEL_PATH = MODEL_CACHE
    print(f"Model cached: {MODEL_PATH}")
else:
    print("Downloading base model ~60GB (~50min first time)...")
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

# ===== STEP 4: Model + LoRA =====
print("\n" + "=" * 70)
print("STEP 4/7: Load + LoRA (FIX 2: attn=eager, FIX 3: 8 targets NO lm_head)")
print("=" * 70)

from unsloth import FastLanguageModel
print(f"Loading via Unsloth (attn=eager, max_seq_len={MAX_SEQ_LEN})...")
t_load = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False, load_in_8bit=False, full_finetuning=False,
    trust_remote_code=True, unsloth_force_compile=False,
    attn_implementation="eager",  # FIX 2
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded in {time.time()-t_load:.1f}s")

# FIX 3: 8 targets NO lm_head (dgxchen v7)
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj",
]
print(f"LoRA r=32 alpha=32 dropout=0.0, targets ({len(target_modules)}, NO lm_head): {target_modules}")
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0,
    target_modules=target_modules, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
model.print_trainable_parameters()
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
used_gb = (torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]) / 1024**3
print(f"After LoRA: used={used_gb:.1f}GB free={free_gb:.1f}GB")

# ===== STEP 5: SFT records =====
print("\n" + "=" * 70)
print("STEP 5/7: Build SFT records + stratified sampler")
print("=" * 70)

from datasets import Dataset as HFDataset
from torch.utils.data import DataLoader, Sampler
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig

PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
records, record_types = [], []
for _, row in train_df.iterrows():
    cot = str(row.get("generated_cot", ""))
    if not cot or cot == "nan" or len(cot.strip()) < 5: continue
    cot_clean = re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
    user_content = str(row["prompt"]) + PROMPT_SUFFIX
    assistant_content = cot_clean + f"\n</think>\n\\boxed{{{row['answer']}}}"
    records.append({"messages": [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(str(row.get("type", "unknown")))
print(f"SFT records: {len(records)}")
dataset = HFDataset.from_list(records)

# FIX 7: formatting_func com conversation wrap (dgxchen v7)
def formatting_prompts_func(example):
    messages = example["messages"]
    conversations = [messages] if messages and isinstance(messages[0], dict) else messages
    texts = []
    for conv in conversations:
        try:
            t = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        except TypeError:
            t = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        texts.append(t)
    return texts

def build_stratified_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for i, l in enumerate(labels): by_label[l].append(i)
    rng = random.Random(seed)
    for v in by_label.values(): rng.shuffle(v)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    order = list(range(n_batches)); rng.shuffle(order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[order[assigned % n_batches]].append(idx)
            assigned += 1
    return [i for b in batches for i in b]

class OrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

class StratSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order
    def get_train_dataloader(self):
        if self.stratified_order is None: return super().get_train_dataloader()
        dk = {"batch_size": self.args.per_device_train_batch_size,
              "sampler": OrderSampler(self.stratified_order),
              "collate_fn": self.data_collator,
              "num_workers": self.args.dataloader_num_workers,
              "pin_memory": self.args.dataloader_pin_memory,
              "persistent_workers": self.args.dataloader_persistent_workers,
              "drop_last": self.args.dataloader_drop_last}
        if self.args.dataloader_num_workers > 0:
            dk["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dk)

class HealthGateCallback(TrainerCallback):
    def __init__(self):
        self.loss_hist = deque(maxlen=10); self.grad_hist = deque(maxlen=5)
        self.min_loss = float("inf"); self.steps_no_improve = 0
        self.high_grad = 0; self.t0 = None
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        print("=" * 70 + "\nV80 HEALTH GATES ATIVOS\n" + "=" * 70)
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        step = int(state.global_step)
        loss = logs.get("loss"); grad = logs.get("grad_norm")
        lr = logs.get("learning_rate"); epoch = logs.get("epoch", 0.0)
        if loss is not None and isinstance(loss, float):
            if math.isnan(loss) or math.isinf(loss):
                print(f"!!! NaN/Inf step {step} ABORT"); control.should_training_stop = True; return
            if loss > 30.0 and step > 10:
                print(f"!!! loss explosion {loss:.3f} step {step} ABORT"); control.should_training_stop = True; return
            self.loss_hist.append(float(loss))
            if float(loss) < self.min_loss: self.min_loss = float(loss); self.steps_no_improve = 0
            else: self.steps_no_improve += 5
        if grad is not None:
            self.grad_hist.append(float(grad))
            self.high_grad = self.high_grad + 1 if float(grad) > 50.0 else 0
            if self.high_grad >= 3: print(f"!!! grad>50 3x (no clipping)")
        free = torch.cuda.mem_get_info()[0] / 1024**3
        total = torch.cuda.mem_get_info()[1] / 1024**3
        used = total - free; peak = torch.cuda.max_memory_allocated() / 1024**3
        if free < 2.0: print(f"!!! VRAM free={free:.1f}GB OOM risk")
        el = time.time() - self.t0 if self.t0 else 0
        total_s = state.max_steps if state.max_steps else 1
        eta = (el / max(step, 1)) * (total_s - step) if step > 0 else 0
        avg_l = sum(self.loss_hist)/len(self.loss_hist) if self.loss_hist else 0
        avg_g = sum(self.grad_hist)/len(self.grad_hist) if self.grad_hist else 0
        _l = float(loss) if loss is not None else 0.0
        _g = float(grad) if grad is not None else 0.0
        _r = float(lr) if lr is not None else 0.0
        print(f"[step {step:4d}/{total_s:4d} ep{epoch:.2f} {step/max(total_s,1)*100:5.1f}%] "
              f"loss={_l:.4f} avg10={avg_l:.4f} grad={_g:.3f} avg5={avg_g:.3f} "
              f"lr={_r:.2e} vram={used:.1f}/{peak:.1f}/{free:.1f}GB "
              f"elapsed={int(el//60)}m ETA={int(eta//60)}m")
    def on_train_end(self, args, state, control, **kw):
        el = time.time() - self.t0 if self.t0 else 0
        print(f"=" * 70 + f"\nTRAIN DONE elapsed={el/60:.1f}min min_loss={self.min_loss:.4f}\n" + "=" * 70)

# ===== STEP 6: Training =====
print("\n" + "=" * 70)
print("STEP 6/7: Train 1 epoch (dgxchen v7 EXACT + FIX 14: paged_adamw_8bit)")
print("=" * 70)

OUT_DIR = "/content/kg1_out/sft_v80"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,                          # FIX 4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=0,
    max_length=MAX_SEQ_LEN,                      # FIX 12: 3072
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,                           # FIX 5
    optim="paged_adamw_8bit",                    # FIX 14: 8bit optimizer (-5GB VRAM)
    logging_steps=5, logging_first_step=True,
    save_strategy="steps", save_steps=50, save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,                 # FIX 6
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,                    # FIX 10
    remove_unused_columns=False,
    seed=SEED, report_to="none", packing=False,
)

order = build_stratified_order(record_types, 32, SEED)
print(f"Eff batch: 32  Total optim steps: {math.ceil(len(record_types)/32)}")

trainer = StratSFTTrainer(
    model=model, args=training_args, train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,     # FIX 7
    stratified_order=order,
    callbacks=[HealthGateCallback()],
)

print("Starting V80 SFT...")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
print(f"Training done: {(time.time()-t0)/60:.1f} min")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f}GB")

# Save adapter
ADAPTER_DIR = "/content/kg1_adapter_v80"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved: {ADAPTER_DIR}")

# ===== STEP 7: Submit =====
print("\n" + "=" * 70)
print("STEP 7/7: Submission.zip + HF upload + Kaggle submit")
print("=" * 70)

BASE = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SUB_DIR = "/content/kg1_out/submission_v80"
Path(SUB_DIR).mkdir(parents=True, exist_ok=True)
required = ["adapter_config.json", "adapter_model.safetensors"]
for fn in required:
    shutil.copy2(Path(ADAPTER_DIR)/fn, Path(SUB_DIR)/fn)
    print(f"Copied {fn} ({(Path(SUB_DIR)/fn).stat().st_size/1024**2:.1f}MB)")

cfg_path = Path(SUB_DIR) / "adapter_config.json"
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
tm = cfg.get("target_modules", [])
print(f"adapter targets: {len(tm) if isinstance(tm, list) else 'parameter list'}, lm_head present: {'lm_head' in tm if isinstance(tm, list) else 'N/A'}")

zip_path = "/content/kg1_out/submission_v80.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in required:
        zf.write(Path(SUB_DIR)/fn, arcname=fn)
print(f"zip: {os.path.getsize(zip_path)/1024**2:.1f}MB")

# HF upload
try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    REPO = "felipesp1983/kg1-nemotron-lora-v80-final"
    api.create_repo(REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=REPO, folder_path=SUB_DIR,
                      allow_patterns=["adapter_*"], token=HF_TOKEN)
    print(f"HF: https://huggingface.co/{REPO}")
except Exception as e:
    print(f"HF upload failed (non-fatal): {e}")

# Kaggle submit (slot check)
try:
    rc = subprocess.run(["kaggle", "competitions", "submissions",
                         "-c", "nvidia-nemotron-model-reasoning-challenge", "--csv"],
                        capture_output=True, text=True, timeout=60)
    if rc.returncode == 0:
        from io import StringIO; import csv as _csv
        today = datetime.datetime.now().strftime("%Y-%m-%d")
        cnt = sum(1 for r in _csv.DictReader(StringIO(rc.stdout)) if r.get("date","").startswith(today))
        print(f"Submissions today: {cnt}/5")
        if cnt < 5:
            msg = f"V80 FINAL dgxchen v7 EXACT {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}"
            r = subprocess.run(["kaggle", "competitions", "submit",
                                "-c", "nvidia-nemotron-model-reasoning-challenge",
                                "-f", zip_path, "-m", msg],
                               capture_output=True, text=True, timeout=600)
            print(f"Submit rc={r.returncode}\n{r.stdout[-400:]}")
        else:
            print(f"Slot full. Manual: kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge -f {zip_path} -m V80")
except Exception as e:
    print(f"Submit fail: {e}")

print("\n" + "=" * 70)
print("V80 FINAL DONE - check score at")
print("https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions")
print("=" * 70)
'''

with open('/content/v80_final.py', 'w', encoding='utf-8') as f:
    f.write(TRAIN_SCRIPT)
print(f'Training script written to /content/v80_final.py ({os.path.getsize("/content/v80_final.py")/1024:.1f} KB)')

# ===== PART 5: RUN TRAINING =====
print()
print('=' * 70)
print('Starting V80 FINAL training (ETA ~2.5-3h)')
print('All 14 fixes applied:')
print('  1. Dataset problem_ids_matched.csv')
print('  2. attn_implementation=eager')
print('  3. LoRA 8 targets NO lm_head')
print('  4. num_train_epochs=1')
print('  5. max_grad_norm=1e9')
print('  6. gradient_checkpointing=True + use_reentrant=False')
print('  7. formatting_func trainer + conversation wrap')
print('  8. mamba-ssm + causal-conv1d wheels')
print('  9. torch 2.5.1 via wheels diretos')
print(' 10. dataloader_num_workers=0')
print(' 11. Unsloth --no-deps + uninstall torchcodec')
print(' 12. MAX_SEQ_LEN=3072 (p99 safe)')
print(' 13. PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True')
print(' 14. optim=paged_adamw_8bit (-5GB VRAM, no offload, 3-4x speedup)')
print('=' * 70)
print()

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # FIX 13
env['MAX_SEQ_LEN'] = '3072'  # FIX 12

proc = subprocess.Popen(
    [sys.executable, '-u', '/content/v80_final.py'],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.wait()

print()
print('=' * 70)
print(f'V80 FINAL DONE - rc={proc.returncode}')
print('=' * 70)

V80 MEGA FINAL - 14 fixes consolidados
HF token: ...ifYYkxHG
Kaggle user: felipe1983
GPU: NVIDIA H100 80GB HBM3 79.2GB

Installing dependencies (torch 2.5.1 + mamba-ssm + ML stack)...
  Uninstalling torch + Colab pre-installed torchcodec/torchao/torchdata...
  Installing torch 2.5.1+cu124 (direct wheels)...
  Installing mamba-ssm + causal-conv1d (torch 2.5 ABI wheels)...
  Installing transformers/peft/trl/accelerate/datasets/bitsandbytes...
  Installing unsloth + unsloth_zoo (--no-deps)...

Verifying child process (torch 2.5 clean import)...
child: torch=2.5.1+cu124 cuda=12.4
child: mamba_ssm=2.2.4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
child: ALL IMPORTS OK

Training script written to /content/v80_final.py (14.9 KB)

Starting V80 FINAL training (ETA ~2.5-3h)
All 14 fixes applied:
  1. Dataset problem_ids_matched.csv
  2. attn_implementation=eager
  3. LoRA 8 targets NO lm_head
  4. num_t